# generating user notifications

Cell 1: Core Helper Functions
This cell defines the logic for distributing timestamps across the user's optimal time windows, picking the right channels, and handling the templates

In [2]:
import pandas as pd
import numpy as np
import random
import ast
from datetime import timedelta

# ==========================================
# 1. Configuration & Mappings
# ==========================================
WINDOW_BOUNDS = {
    'early_morning': (6, 9),
    'mid_morning': (9, 12),
    'afternoon': (12, 15),
    'late_afternoon': (15, 18),
    'evening': (18, 21),
    'night': (21, 24)
}

# ==========================================
# 2. Helper Functions
# ==========================================
def generate_times_for_windows(windows: list, num_notifs: int) -> list:
    """
    Distributes `num_notifs` across the available `windows` and generates
    spaced-out exact timestamps (HH:MM) in chronological order.
    """
    if not windows:
        windows = ['afternoon', 'evening'] # Fallback
        
    # Sort windows chronologically to ensure natural progression
    windows = sorted(windows, key=lambda w: WINDOW_BOUNDS[w][0])
    
    # Calculate how many notifications go into each window
    base_count = num_notifs // len(windows)
    remainder = num_notifs % len(windows)
    
    distribution = [base_count] * len(windows)
    # Give the remainder to the later windows (evening/night) where users are typically more active
    for i in range(remainder):
        distribution[-(i + 1)] += 1
        
    times_in_minutes = []
    
    for i, window in enumerate(windows):
        start_hour, end_hour = WINDOW_BOUNDS[window]
        count = distribution[i]
        
        if count == 0:
            continue
            
        # Divide the window into `count` chunks to ensure spacing
        chunk_size = ((end_hour - start_hour) * 60) // count
        
        for j in range(count):
            chunk_start = (start_hour * 60) + (j * chunk_size)
            chunk_end = chunk_start + chunk_size - 1
            # Pick a random minute within this chunk
            rand_minute = random.randint(chunk_start, chunk_end)
            times_in_minutes.append(rand_minute)
            
    # Sort just in case, then format to HH:MM
    times_in_minutes.sort()
    formatted_times = [
        f"{str(m // 60).zfill(2)}:{str(m % 60).zfill(2)}" 
        for m in times_in_minutes
    ]
    return formatted_times

def assign_channel(sequence_index: int, total_notifs: int) -> str:
    """
    Decides the channel based on the notification's position in the day's sequence.
    """
    if sequence_index == 0:
        # First interaction of the day: Email for planning/recap, or Push
        return random.choice(['Email', 'Push'])
    elif sequence_index == total_notifs - 1:
        # Last interaction: Always Push (urgency, streak warnings)
        return 'Push'
    else:
        # Middle interactions: Mix of In-App (if they opened it) and Push
        return random.choice(['Push', 'In-App', 'Push'])

Cell 2: The Schedule Generator Engine
This cell loads the assigned frequencies, the ML-generated optimal windows, and the templates, merging them into the final wide-format schedule table.

In [3]:
# ==========================================
# 3. Main Schedule Generator Execution
# ==========================================
def generate_schedules(path_users, path_gmm, path_templates, path_output):
    print("Loading datasets...")
    
    # 1. Load Data
    try:
        # Assuming your frequency CSV has user_id, segment_id, lifecycle_stage, daily_notification_limit
        df_users = pd.read_csv(path_users) 
        df_gmm = pd.read_csv(path_gmm)
        df_templates = pd.read_csv(path_templates)
    except FileNotFoundError as e:
        print(f"Error loading files: {e}. Please check your ./output/ directory.")
        return

    # Mock lifecycle_stage and lifecycle_day if they don't exist in your user dataset yet
    if 'lifecycle_stage' not in df_users.columns:
        df_users['lifecycle_stage'] = random.choices(['trial', 'paid', 'churned', 'inactive'], k=len(df_users))
    if 'lifecycle_day' not in df_users.columns:
        df_users['lifecycle_day'] = np.random.randint(1, 30, size=len(df_users))

    # 2. Merge Users with their Optimal Windows
    # Convert string representation of list "['morning', 'night']" back to an actual Python list
    df_gmm['optimal_windows'] = df_gmm['optimal_windows'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    df_merged = pd.merge(df_users, df_gmm[['segment_id', 'optimal_windows']], on='segment_id', how='left')

    # 3. Group templates for fast lookup
    # Creates a dict: {('S1', 'trial'): [df_of_templates], ...}
    template_pool = {}
    for (seg, stage), group in df_templates.groupby(['segment_id', 'lifecycle_stage']):
        template_pool[(seg, stage)] = group['template_id'].tolist()

    print(f"Generating schedules for {len(df_merged)} users...")
    
    # 4. Build the Schedule
    schedules = []
    
    for _, user in df_merged.iterrows():
        user_id = user.get('user_id', f"U_{random.randint(1000, 9999)}") # Fallback ID
        segment = user['segment_id']
        stage = user['lifecycle_stage']
        limit = int(user.get('daily_notification_limit', 4))
        windows = user.get('optimal_windows', ['afternoon', 'evening'])
        
        # Guard against NaN windows
        if not isinstance(windows, list):
            windows = ['afternoon', 'evening']

        # Get exact timestamps for today
        times = generate_times_for_windows(windows, limit)
        
        # Get available templates for this user's profile
        available_templates = template_pool.get((segment, stage), [])
        
        # Base dictionary with user metadata
        row = {
            'user_id': user_id,
            'segment_id': segment,
            'lifecycle_stage': stage,
            'lifecycle_day': user['lifecycle_day'],
            'daily_limit': limit
        }
        
        # Fill the wide format columns for slots 1 through 9
        for i in range(9):
            col_idx = i + 1
            if i < limit:
                # Assign Time
                row[f'notif_{col_idx}_time'] = times[i]
                
                # Assign Template (Randomly sample without replacement if possible, otherwise reuse)
                if len(available_templates) >= limit:
                    # Deduplication guaranteed for the day
                    selected_template = random.sample(available_templates, limit)[i]
                elif len(available_templates) > 0:
                    selected_template = random.choice(available_templates)
                else:
                    selected_template = "TPL-DEFAULT"
                    
                row[f'notif_{col_idx}_template_id'] = selected_template
                
                # Assign Channel
                row[f'notif_{col_idx}_channel'] = assign_channel(i, limit)
            else:
                # Null out the columns if the slot is beyond the user's daily limit
                row[f'notif_{col_idx}_time'] = None
                row[f'notif_{col_idx}_template_id'] = None
                row[f'notif_{col_idx}_channel'] = None
                
        schedules.append(row)

    # 5. Export Final Table
    df_schedule = pd.DataFrame(schedules)
    df_schedule.to_csv(path_output, index=False)
    print(f"\nSuccess! User schedule generated and saved to: {path_output}")
    
    # Display a sneak peek of the first user
    print("\nSneak peek of wide-format schedule (User 1):")
    cols_to_show = ['user_id', 'daily_limit', 'notif_1_time', 'notif_1_template_id', 'notif_1_channel', 'notif_2_time']
    print(df_schedule[cols_to_show].head(1).to_string(index=False))

# ==========================================
# Run the pipeline
# ==========================================
if __name__ == "__main__":
    PATH_USERS = './output/users_frequency_assigned.csv' # Assuming you named it this in the previous step
    PATH_GMM_WINDOWS = './output/users_segmented_GMM.csv'
    PATH_TEMPLATES = './output/message_templates.csv'
    PATH_OUTPUT = './output/user_schedules_final.csv'
    
    generate_schedules(PATH_USERS, PATH_GMM_WINDOWS, PATH_TEMPLATES, PATH_OUTPUT)

Loading datasets...
Generating schedules for 61 users...

Success! User schedule generated and saved to: ./output/user_schedules_final.csv

Sneak peek of wide-format schedule (User 1):
user_id  daily_limit notif_1_time notif_1_template_id notif_1_channel notif_2_time
   US_1            7        06:08        TPL-5984019C           Email        07:38


# Iteration - 2

In [6]:
import pandas as pd
import numpy as np

# Define paths
PATH_RESULTS = './output/experiment_results.csv'
PATH_TEMPLATES = './output/message_templates.csv'
PATH_SCORED_TEMPLATES = './output/message_templates_scored.csv'

# ==========================================
# 1. Core Classification Logic
# ==========================================
def classify_template(ctr: float, eng: float):
    if pd.isna(ctr) or pd.isna(eng):
        return pd.Series(["UNTESTED", "No Data", True, 1.0])

    # CTR Score
    ctr_score = 2 if ctr > 15.0 else (1 if ctr >= 5.0 else 0)
    
    # Engagement Score
    eng_score = 2 if eng > 40.0 else (1 if eng >= 20.0 else 0)

    total_score = ctr_score + eng_score

    # Classifications & Tags
    if total_score == 4:
        classification = "GOOD"
        tag = "Champion (Scale This)"
    elif total_score >= 2:
        classification = "NEUTRAL"
        if ctr_score == 2 and eng_score == 0:
            tag = "Clickbait (Great Hook, Poor Relevance)"
        elif ctr_score == 0 and eng_score == 2:
            tag = "Hidden Gem (Boring Hook, Great Relevance)"
        else:
            tag = "Average Performer"
    else: # Total 0 or 1
        classification = "BAD"
        tag = "Retire Template"

    # Feedback Loop Variables
    is_active = False if classification == "BAD" else True
    # Up-weight Champions so the Schedule Generator picks them more often
    weight_multiplier = 2.0 if classification == "GOOD" else 1.0

    return pd.Series([classification, tag, is_active, weight_multiplier])

# ==========================================
# 2. Execution Pipeline
# ==========================================
print(f"Loading experiment data from {PATH_RESULTS}...")
df_results = pd.read_csv(PATH_RESULTS)

# Calculate percentages mathematically
df_results['ctr_percent'] = (df_results['clicks'] / df_results['sends']) * 100
df_results['engagement_percent'] = np.where(
    df_results['clicks'] > 0, 
    (df_results['engagements'] / df_results['clicks']) * 100, 
    0.0
)

# Apply classification row-by-row
df_results[['performance_class', 'performance_tag', 'is_active', 'weight_multiplier']] = df_results.apply(
    lambda row: classify_template(row['ctr_percent'], row['engagement_percent']), 
    axis=1
)

# Load the main templates database
try:
    df_templates = pd.read_csv(PATH_TEMPLATES)
except FileNotFoundError:
    print(f"Error: Could not find {PATH_TEMPLATES}.")
    raise

# Merge the new performance data with the templates
merge_cols = ['template_id', 'ctr_percent', 'engagement_percent', 'performance_class', 'performance_tag', 'is_active', 'weight_multiplier']
df_scored = pd.merge(df_templates, df_results[merge_cols], on='template_id', how='left')

# Handle Untested Templates (Templates that haven't been sent yet)
df_scored['performance_class'] = df_scored['performance_class'].fillna('UNTESTED')
df_scored['performance_tag'] = df_scored['performance_tag'].fillna('No Data Yet')
df_scored['is_active'] = df_scored['is_active'].fillna(True) # Keep untested templates active
df_scored['weight_multiplier'] = df_scored['weight_multiplier'].fillna(1.0) # Standard weight

# Export the updated, scored templates file
df_scored.to_csv(PATH_SCORED_TEMPLATES, index=False)
print(f"\nSuccessfully generated scored templates at: {PATH_SCORED_TEMPLATES}")

# Summary Output
print("\n--- CLASSIFICATION SUMMARY ---")
print(df_scored['performance_class'].value_counts())

print("\n--- SNEAK PEEK (Scored Templates) ---")
display_cols = ['template_id', 'performance_class', 'performance_tag', 'is_active', 'weight_multiplier']
print(df_scored[display_cols].head(6).to_string(index=False))

Loading experiment data from ./output/experiment_results.csv...

Successfully generated scored templates at: ./output/message_templates_scored.csv

--- CLASSIFICATION SUMMARY ---
performance_class
UNTESTED    349
NEUTRAL       3
BAD           2
GOOD          1
Name: count, dtype: int64

--- SNEAK PEEK (Scored Templates) ---
 template_id performance_class                           performance_tag is_active  weight_multiplier
TPL-CC906E7D              GOOD                     Champion (Scale This)      True                2.0
TPL-4A53BADC               BAD                           Retire Template     False                1.0
TPL-ED34FF29           NEUTRAL    Clickbait (Great Hook, Poor Relevance)      True                1.0
TPL-5CA99153           NEUTRAL Hidden Gem (Boring Hook, Great Relevance)      True                1.0
TPL-FA4A92E2           NEUTRAL                         Average Performer      True                1.0
TPL-B4D5946E               BAD                           Retir

## remaking the users results in iteration 2

In [9]:
import pandas as pd

# ==========================================
# Configuration & File Paths
# ==========================================
PATH_SCORED_TEMPLATES = './output/message_templates_scored.csv'
PATH_WEIGHTED_POOL = './output/message_templates_weighted_pool.csv'

# ==========================================
# Execution: Adjusting Template Counts
# ==========================================
def adjust_template_counts():
    print(f"Loading scored templates from {PATH_SCORED_TEMPLATES}...")
    try:
        df_scored = pd.read_csv(PATH_SCORED_TEMPLATES)
    except FileNotFoundError:
        print("Scored templates not found. Run the classification pipeline first.")
        return

    # 1. Define the Count Multipliers
    # GOOD = Appears 3 times (Highly likely to be picked)
    # NEUTRAL / UNTESTED = Appears 1 time (Standard chance)
    # BAD = Appears 0 times (Completely removed from the pool)
    def determine_count(row):
        perf_class = row.get('performance_class', 'UNTESTED')
        
        if perf_class == 'GOOD':
            return 3
        elif perf_class == 'BAD':
            return 0
        else:
            return 1 # Covers NEUTRAL and UNTESTED

    # Apply the logic to get a specific count for each row
    df_scored['target_count'] = df_scored.apply(determine_count, axis=1)

    # 2. Physically expand the dataframe based on the target count
    # This elegant Pandas trick duplicates the row 'target_count' times
    df_weighted_pool = df_scored.loc[df_scored.index.repeat(df_scored['target_count'])].reset_index(drop=True)

    # Clean up the helper column
    df_weighted_pool = df_weighted_pool.drop(columns=['target_count'])

    # 3. Export the new weighted pool
    df_weighted_pool.to_csv(PATH_WEIGHTED_POOL, index=False)
    
    print("\n--- TEMPLATE POOL ADJUSTMENT SUMMARY ---")
    print(f"Original unique templates: {len(df_scored)}")
    print(f"Templates removed (BAD): {len(df_scored[df_scored['performance_class'] == 'BAD'])}")
    print(f"Templates duplicated (GOOD): {len(df_scored[df_scored['performance_class'] == 'GOOD'])}")
    print(f"Total rows in new weighted pool: {len(df_weighted_pool)}")
    print(f"\nSuccess! The Schedule Generator should now pull from: {PATH_WEIGHTED_POOL}")

# Run the adjustment
adjust_template_counts()

Loading scored templates from ./output/message_templates_scored.csv...

--- TEMPLATE POOL ADJUSTMENT SUMMARY ---
Original unique templates: 358
Templates removed (BAD): 2
Templates duplicated (GOOD): 1
Total rows in new weighted pool: 358

Success! The Schedule Generator should now pull from: ./output/message_templates_weighted_pool.csv


make the delta reporter thing

In [11]:
import pandas as pd
import numpy as np

# ==========================================
# Configuration & File Paths
# ==========================================
PATH_SCORED_TEMPLATES = './output/message_templates_scored.csv'
PATH_GMM_WINDOWS = './output/users_segmented_GMM.csv'
PATH_DELTA_REPORT = './output/delta_report.csv'

# ==========================================
# Delta Reporter Class
# ==========================================
class DeltaReporter:
    def __init__(self):
        self.audit_log = []

    def log_change(self, entity_type, entity_id, change_type, metric_trigger, before_value, after_value, explanation):
        self.audit_log.append({
            'entity_type': entity_type,
            'entity_id': entity_id,
            'change_type': change_type,
            'metric_trigger': metric_trigger,
            'before_value': before_value,
            'after_value': after_value,
            'explanation': explanation
        })

    def export_report(self, path):
        df_report = pd.DataFrame(self.audit_log)
        if not df_report.empty:
            df_report.to_csv(path, index=False)
            print(f"✅ Delta Report successfully exported to: {path}")
        else:
            print("⚠️ No changes to report.")
        return df_report

# ==========================================
# Execution Engine
# ==========================================
def run_delta_analysis():
    reporter = DeltaReporter()
    
    print("Analyzing system changes for Delta Report...")

    # ---------------------------------------------------------
    # 1. Template Classification Deltas (The Content Changes)
    # ---------------------------------------------------------
    try:
        df_templates = pd.read_csv(PATH_SCORED_TEMPLATES)
        
        # Calculate Expected System CTR Before (Unweighted Average)
        # Using a default of 5% for untested templates to prevent NaN issues
        original_ctr_mean = df_templates['ctr_percent'].fillna(5.0).mean()
        
        # Calculate Expected System CTR After (Weighted Average based on Learning Engine)
        weighted_ctrs = []
        for _, row in df_templates.iterrows():
            ctr = row['ctr_percent'] if not pd.isna(row['ctr_percent']) else 5.0
            weight = row.get('target_count', 1) # Assumes you ran the weighting step
            if 'target_count' not in row:
                 # Fallback logic if weighting script wasn't saved directly to this file
                 weight = 3 if row['performance_class'] == 'GOOD' else (0 if row['performance_class'] == 'BAD' else 1)
                 
            weighted_ctrs.extend([ctr] * weight)
            
            # Log the individual template changes
            if row['performance_class'] == 'BAD':
                reporter.log_change(
                    entity_type="Template",
                    entity_id=row['template_id'],
                    change_type="Suppression",
                    metric_trigger=f"CTR: {row['ctr_percent']:.1f}%, ENG: {row['engagement_percent']:.1f}%",
                    before_value="Weight: 1x (Active)",
                    after_value="Weight: 0x (Inactive)",
                    explanation="Template failed to meet minimum engagement thresholds. Suppressed to protect daily active minutes and avoid user fatigue."
                )
            elif row['performance_class'] == 'GOOD':
                reporter.log_change(
                    entity_type="Template",
                    entity_id=row['template_id'],
                    change_type="Promotion",
                    metric_trigger=f"CTR: {row['ctr_percent']:.1f}%, ENG: {row['engagement_percent']:.1f}%",
                    before_value="Weight: 1x",
                    after_value="Weight: 3x",
                    explanation="Template exceeded top-tier thresholds. Oversampled mathematically to scale the winning psychological hook across the segment."
                )

        new_ctr_mean = np.mean(weighted_ctrs) if weighted_ctrs else original_ctr_mean
        ctr_uplift = new_ctr_mean - original_ctr_mean

    except FileNotFoundError:
        print(f"Warning: {PATH_SCORED_TEMPLATES} not found. Skipping template deltas.")
        original_ctr_mean, new_ctr_mean, ctr_uplift = 0, 0, 0

    # ---------------------------------------------------------
    # 2. Timing Optimization Deltas (The GMM Changes)
    # ---------------------------------------------------------
    try:
        df_gmm = pd.read_csv(PATH_GMM_WINDOWS)
        
        for _, row in df_gmm.iterrows():
            # Assume Iteration 1 used a standard static broadcast window
            static_baseline = "['afternoon', 'evening']"
            new_windows = str(row['optimal_windows'])
            
            if static_baseline != new_windows:
                reporter.log_change(
                    entity_type="Segment Timing",
                    entity_id=row['segment_id'],
                    change_type="Window Shift",
                    metric_trigger=f"Lowest BIC Score (K={row.get('num_windows_assigned', 'Auto')})",
                    before_value=static_baseline,
                    after_value=new_windows,
                    explanation=f"GMM algorithm detected unique activity clusters for this segment. Adjusted sending windows to align with localized biological/habitual prime times."
                )
    except FileNotFoundError:
        print(f"Warning: {PATH_GMM_WINDOWS} not found. Skipping timing deltas.")

    # ---------------------------------------------------------
    # 3. Export and Display Measurable Impact
    # ---------------------------------------------------------
    df_report = reporter.export_report(PATH_DELTA_REPORT)
    
    print("\n" + "="*50)
    print("📈 MEASURABLE DELTA (ITERATION 1 vs ITERATION 2)")
    print("="*50)
    print(f"Original Expected System CTR:  {original_ctr_mean:.2f}%")
    print(f"New Expected System CTR:       {new_ctr_mean:.2f}%")
    print(f"Projected Performance Uplift: +{ctr_uplift:.2f}% absolute improvement")
    print("="*50)
    
    if not df_report.empty:
        print("\nSneak Peek of Delta Audit Log:")
        print(df_report[['entity_type', 'change_type', 'metric_trigger', 'explanation']].head().to_string(index=False))

# Run the Reporter
run_delta_analysis()

Analyzing system changes for Delta Report...
✅ Delta Report successfully exported to: ./output/delta_report.csv

📈 MEASURABLE DELTA (ITERATION 1 vs ITERATION 2)
Original Expected System CTR:  5.09%
New Expected System CTR:       5.16%
Projected Performance Uplift: +0.07% absolute improvement

Sneak Peek of Delta Audit Log:
   entity_type  change_type         metric_trigger                                                                                                                                         explanation
      Template    Promotion CTR: 18.0%, ENG: 45.0%                       Template exceeded top-tier thresholds. Oversampled mathematically to scale the winning psychological hook across the segment.
      Template  Suppression  CTR: 4.0%, ENG: 15.0%                           Template failed to meet minimum engagement thresholds. Suppressed to protect daily active minutes and avoid user fatigue.
      Template  Suppression  CTR: 6.0%, ENG: 15.0%                           T

In [10]:
import pandas as pd
import numpy as np
import random
import ast
from datetime import timedelta

# ==========================================
# 1. Configuration & Mappings
# ==========================================
WINDOW_BOUNDS = {
    'early_morning': (6, 9),
    'mid_morning': (9, 12),
    'afternoon': (12, 15),
    'late_afternoon': (15, 18),
    'evening': (18, 21),
    'night': (21, 24)
}

# ==========================================
# 2. Helper Functions
# ==========================================
def generate_times_for_windows(windows: list, num_notifs: int) -> list:
    if not windows:
        windows = ['afternoon', 'evening']
        
    windows = sorted(windows, key=lambda w: WINDOW_BOUNDS[w][0])
    
    base_count = num_notifs // len(windows)
    remainder = num_notifs % len(windows)
    
    distribution = [base_count] * len(windows)
    for i in range(remainder):
        distribution[-(i + 1)] += 1
        
    times_in_minutes = []
    
    for i, window in enumerate(windows):
        start_hour, end_hour = WINDOW_BOUNDS[window]
        count = distribution[i]
        
        if count == 0:
            continue
            
        chunk_size = ((end_hour - start_hour) * 60) // count
        
        for j in range(count):
            chunk_start = (start_hour * 60) + (j * chunk_size)
            chunk_end = chunk_start + chunk_size - 1
            rand_minute = random.randint(chunk_start, chunk_end)
            times_in_minutes.append(rand_minute)
            
    times_in_minutes.sort()
    formatted_times = [
        f"{str(m // 60).zfill(2)}:{str(m % 60).zfill(2)}" 
        for m in times_in_minutes
    ]
    return formatted_times

def assign_channel(sequence_index: int, total_notifs: int) -> str:
    if sequence_index == 0:
        return random.choice(['Email', 'Push'])
    elif sequence_index == total_notifs - 1:
        return 'Push'
    else:
        return random.choice(['Push', 'In-App', 'Push'])


# ==========================================
# 3. Main Schedule Generator Execution
# ==========================================
def generate_schedules(path_users, path_gmm, path_templates, path_output):
    print(f"Loading datasets (Using weighted templates: {path_templates})...")
    
    try:
        df_users = pd.read_csv(path_users) 
        df_gmm = pd.read_csv(path_gmm)
        df_templates = pd.read_csv(path_templates)
    except FileNotFoundError as e:
        print(f"Error loading files: {e}. Please check your ./output/ directory.")
        return

    if 'lifecycle_stage' not in df_users.columns:
        df_users['lifecycle_stage'] = random.choices(['trial', 'paid', 'churned', 'inactive'], k=len(df_users))
    if 'lifecycle_day' not in df_users.columns:
        df_users['lifecycle_day'] = np.random.randint(1, 30, size=len(df_users))

    df_gmm['optimal_windows'] = df_gmm['optimal_windows'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    df_merged = pd.merge(df_users, df_gmm[['segment_id', 'optimal_windows']], on='segment_id', how='left')

    # Group templates for fast lookup (this now includes duplicated 'GOOD' templates)
    template_pool = {}
    for (seg, stage), group in df_templates.groupby(['segment_id', 'lifecycle_stage']):
        template_pool[(seg, stage)] = group['template_id'].tolist()

    print(f"Generating Iteration 2 schedules for {len(df_merged)} users...")
    
    schedules = []
    
    for _, user in df_merged.iterrows():
        user_id = user.get('user_id', f"U_{random.randint(1000, 9999)}") 
        segment = user['segment_id']
        stage = user['lifecycle_stage']
        limit = int(user.get('daily_notification_limit', 4))
        windows = user.get('optimal_windows', ['afternoon', 'evening'])
        
        if not isinstance(windows, list):
            windows = ['afternoon', 'evening']

        times = generate_times_for_windows(windows, limit)
        available_templates = template_pool.get((segment, stage), [])
        
        # --- NEW DEDUPLICATION LOGIC FOR WEIGHTED POOL ---
        # We pre-select the templates for the day ensuring no duplicates, while respecting weights
        chosen_templates_for_day = []
        working_pool = available_templates.copy()
        
        for _ in range(limit):
            if not working_pool:
                chosen_templates_for_day.append("TPL-DEFAULT")
                continue
            
            # Pick a template (GOOD templates have a higher mathematical chance of being picked here)
            choice = random.choice(working_pool)
            chosen_templates_for_day.append(choice)
            
            # Strip ALL instances of the chosen template from the working pool to prevent same-day spam
            working_pool = [t for t in working_pool if t != choice]
        # -------------------------------------------------

        row = {
            'user_id': user_id,
            'segment_id': segment,
            'lifecycle_stage': stage,
            'lifecycle_day': user['lifecycle_day'],
            'daily_limit': limit
        }
        
        for i in range(9):
            col_idx = i + 1
            if i < limit:
                row[f'notif_{col_idx}_time'] = times[i]
                row[f'notif_{col_idx}_template_id'] = chosen_templates_for_day[i]
                row[f'notif_{col_idx}_channel'] = assign_channel(i, limit)
            else:
                row[f'notif_{col_idx}_time'] = None
                row[f'notif_{col_idx}_template_id'] = None
                row[f'notif_{col_idx}_channel'] = None
                
        schedules.append(row)

    df_schedule = pd.DataFrame(schedules)
    df_schedule.to_csv(path_output, index=False)
    print(f"\nSuccess! Iteration 2 schedule generated and saved to: {path_output}")
    
    print("\nSneak peek of wide-format schedule (User 1):")
    cols_to_show = ['user_id', 'daily_limit', 'notif_1_time', 'notif_1_template_id', 'notif_1_channel', 'notif_2_time']
    print(df_schedule[cols_to_show].head(1).to_string(index=False))

# ==========================================
# Run the pipeline
# ==========================================
if __name__ == "__main__":
    PATH_USERS = './output/users_frequency_assigned.csv' 
    PATH_GMM_WINDOWS = './output/users_segmented_GMM.csv'
    
    # NEW: Point to the weighted pool created in the classification feedback step
    PATH_TEMPLATES_ITER2 = './output/message_templates_weighted_pool.csv'
    
    # NEW: Export to a new Iteration 2 schedule file
    PATH_OUTPUT_ITER2 = './output/user_schedules_final_iter2.csv'
    
    generate_schedules(PATH_USERS, PATH_GMM_WINDOWS, PATH_TEMPLATES_ITER2, PATH_OUTPUT_ITER2)

Loading datasets (Using weighted templates: ./output/message_templates_weighted_pool.csv)...
Generating Iteration 2 schedules for 61 users...

Success! Iteration 2 schedule generated and saved to: ./output/user_schedules_final_iter2.csv

Sneak peek of wide-format schedule (User 1):
user_id  daily_limit notif_1_time notif_1_template_id notif_1_channel notif_2_time
   US_1            7        06:13        TPL-54F71B34           Email        07:13
